# 초구 예측 — model_06 (2단계: 구종 1단계 보완 = B4 시즌 초 배수)

- 2단계 구종 1단계 보완 (README 5장 2번 "구종 1단계 보완"): 새 정보 없이 기록 쓰는 방식만 바꾸는 1단계 성격 항목을 구종 출발점(B4)에
  - 시즌 초는 1단계에서 구종·위치 어디에도 안 함, 기준점은 위치에만 함 (model_02)
  - **시즌 초**: 새 시즌 첫 경기 전에 지난 시즌까지 누적 기록 무게에 배수(season_f)를 곱함 -> 비시즌 동안 바뀐 배합을 빨리 따라감
    - 근거: model_05 0-1 김민 3월 gap −0.23·직구 −0.38, 원태인 신인 기록 1–50개 구간 gap −0.10
  - **기준점**: 리그 구종 비율을 지난 시즌 전부(지금) / 직전 시즌만 / 투수 동일 가중 / 전날까지 반감기로
  - 연결된 B4 파라미터(반감기·alpha·hand_k)도 함께 다시 고름 -> 2019 B4 log loss 기준
- B4가 바뀌면 LightGBM 출발점이 바뀜 -> 연결된 LightGBM 파라미터만 model_05 1등 주변에서 다시 고름, A 변수는 그대로
- 위치: model_03 그대로 (2단계 동안 고정)
- 0장 = 모델 전 확인: 0-1 주자 신원 (적용 안 함, README 4-9)
- 비교 대상: model_05

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_06"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. 주자 신원 (주자의 타순) 확인 -> 적용 안 함

- 이유: model_05 0-2 사전 확인에서 A 안 주자 고유 몫이 가장 커서(5.3) 다음 순서였지만, 만들기 비쌈 -> model_05에 주자 몫이 얼마나 남았는지 먼저 확인
- 주자 신원 간이 복원: 같은 반 이닝 바로 앞 타자가 지금 그 루에 있는 경우만
  - 앞 타석 마지막 공의 `place_sc`(A 1루·B 2루·C 3루 진출)와 지금 그 루가 찼는지로 판단, 앞 타석이 데이터에 있고 타순이 바로 앞일 때만
  - 경기 기록, 그 타석 앞 정보만 (미래 행 X)
- 확인 대상: model_05 2019 공별 예측 (2018 예측은 학습에 쓴 해라 잔차가 작음 -> 안 씀)
  - 구종: 칸마다 구종별 배수를 2019 안 경기 단위 5겹 교차로 맞춰 log loss 이득 (model_05 0-2-3과 같은 방식)
  - 위치: 같은 그룹으로 실제 − 예측 (실전 H/S/C)

In [2]:
mm = m.sort_values(fc.KEYS)
pa = (mm.groupby(["gameID", "pa_id"], sort=False)  # 타석 표: 타석 시작 상태 + 타석 결과
      .agg(inning=("inning", "first"), tb=("tb", "first"), p0=("pit_no", "min"), batorder=("batorder", "first"),
           b1=("base1", "first"), b2=("base2", "first"), b3=("base3", "first"), place=("place_sc", "last"))
      .reset_index().sort_values(["gameID", "p0"]))
h = pa.groupby(["gameID", "inning", "tb"], sort=False)  # 반 이닝
pa["prev_place"] = h["place"].shift()  # 바로 앞 타석(데이터에 있는) 타자의 최종 위치
pa["prev_order"] = h["batorder"].shift()  # 그 타자의 타순
pa["prev_gap"] = (pa["batorder"] - pa["prev_order"]) % 9  # 1 = 진짜 바로 앞 타자 (사이에 빠진 타석 없음)
BASE = {"A": ("b1", 1), "B": ("b2", 2), "C": ("b3", 3)}  # place_sc -> (이번 타석 시작 때 그 루 열, 루 번호)
pa["prev_base"] = [BASE[p][1] if (g == 1 and p in BASE and r[BASE[p][0]] == 1) else np.nan
                   for p, g, r in zip(pa["prev_place"], pa["prev_gap"], pa[["b1", "b2", "b3"]].to_dict("records"))]  # 바로 앞 타자가 지금 있는 루 (없으면 빈 값)
pa["prev_order_grp"] = pd.cut(pa["prev_order"], [0, 2, 5, 9], labels=["1–2번", "3–5번", "6–9번"]).astype(str)  # 주자 타순 묶음

d = fc.align(pd.read_parquet("results/model_05/per_pitch.parquet"), fp, fc.VAL_YEAR)  # model_05 2019 초구 + 공별 예측
d = d.merge(pa[["gameID", "pa_id", "prev_base", "prev_order_grp", "prev_gap"]], on=["gameID", "pa_id"], how="left", validate="many_to_one")
runner = ((d["base1"] + d["base2"] + d["base3"]) > 0).to_numpy()
r = d[runner]
display(pd.Series({"주자 있는 초구 (2019)": len(r), "바로 앞 타석이 같은 반 이닝·바로 앞 타자": (r["prev_gap"] == 1).mean(),
                   "그 타자가 지금 루에 있음 (간이 복원)": r["prev_base"].notna().mean(),
                   "  그중 1루": r["prev_base"].eq(1).mean()}).round(3).rename("값").to_frame())

,값
주자 있는 초구 (2019),6232.000
바로 앞 타석이 같은 반 이닝·바로 앞 타자,0.958
그 타자가 지금 루에 있음 (간이 복원),0.665
그중 1루,0.573


#### 0-1-1. 구종: model_05에 남은 주자 몫

- 칸마다 구종별 배수 w = (실제 수 + k·평균 예측) / (예측 합 + k·평균 예측)를 학습 겹에서 구해 확인 겹 확률에 곱함 (2019 경기 단위 5겹)
- 이득 = log loss 감소 × 1000 (양수 = 좋아짐), 95% 구간은 경기 단위 부트스트랩

In [3]:
CL, K = CLASSES, len(CLASSES)
P = d[fc.P_TYPE].to_numpy()  # model_05 구종 확률 (n, 9)
y = d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
Y = np.eye(K)[y]
GID = d["gameID"].to_numpy()
ug = np.unique(GID)
fold = pd.Series(np.random.default_rng(0).integers(0, 5, len(ug)), index=ug).reindex(GID).to_numpy()  # 경기마다 겹 번호

def tilt_cf(b, k):  # 칸 b(공마다 문자열)별 구종 배수를 5겹 교차로 -> 공별 log loss 이득 (n,)
    gain = np.zeros(len(d))
    for f in range(5):
        tr, te = np.flatnonzero(fold != f), np.flatnonzero(fold == f)
        O, E = pd.DataFrame(Y[tr]).groupby(b[tr]).sum(), pd.DataFrame(P[tr]).groupby(b[tr]).sum()  # 칸별 실제 수·예측 합
        pi = np.maximum(E.to_numpy() / pd.Series(b[tr]).value_counts().reindex(E.index).to_numpy()[:, None], 1e-6)  # 칸별 평균 예측
        w = pd.DataFrame((O.to_numpy() + k * pi) / (E.to_numpy() + k * pi), index=E.index).reindex(b[te]).fillna(1.0).to_numpy()
        Q = P[te] * w
        Q /= Q.sum(1, keepdims=True)  # 다시 합 1
        gain[te] = np.log(Q[np.arange(len(te)), y[te]]) - np.log(P[te, y[te]])
    return gain

def game_ci(g, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": GID, "d": g}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

rs = (d["base1"].astype(str) + d["base2"].astype(str) + d["base3"].astype(str)).to_numpy()  # 주자 배치 "100" 등
who = np.where(d["prev_base"].notna(), d["prev_base"].astype("Int64").astype(str) + "루|" + d["prev_order_grp"], "아님").astype(str)
CELLS = {"주자 배치 (8)": rs, "주자 배치 × 아웃 (24)": rs + "|" + d["out_cn"].clip(upper=2).astype(str).to_numpy(),
         "직전 타자가 루에 있음 × 그 타순": who, "직전 타자 루 × 타순 × 주자 배치": who + "|" + rs}
rows = []
for nm, b in CELLS.items():
    k, g = max(((k, tilt_cf(b, k)) for k in (30.0, 100.0, 300.0)), key=lambda t: t[1].mean())  # k는 2019 교차 1등
    lo, hi = game_ci(g)
    rows.append({"칸": nm, "칸 수": len(np.unique(b)), "k": k, "이득 ×1000": g.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000,
                 "주자 있는 공만 ×1000": g[runner].mean() * 1000})
display(pd.DataFrame(rows).set_index("칸").round(2))

d["rFAST"] = Y[:, CL.index("FAST")] - P[:, CL.index("FAST")]  # 직구 실제 − 예측
se = lambda x: x.std() / np.sqrt(len(x))
print("1루 주자 = 직전 타자: 타순별 직구 실제 − 예측")
display(d[d["prev_base"].eq(1)].groupby("prev_order_grp")["rFAST"].agg(n="size", 평균="mean", 표준오차=se).round(3))
print("주자 배치별 직구 실제 − 예측")
display(d.assign(rs=rs).groupby("rs")["rFAST"].agg(n="size", 평균="mean", 표준오차=se).round(3).T)

,칸 수,k,이득 ×1000,95% 하한,95% 상한,주자 있는 공만 ×1000
칸,,,,,,
주자 배치 (8),8,300.0,0.53,-0.70,1.89,0.04
주자 배치 × 아웃 (24),24,300.0,0.55,-0.66,1.87,0.42
직전 타자가 루에 있음 × 그 타순,10,300.0,-0.09,-1.18,1.07,-0.89
직전 타자 루 × 타순 × 주자 배치,32,300.0,0.51,-0.75,1.85,-0.05


1루 주자 = 직전 타자: 타순별 직구 실제 − 예측


,n,평균,표준오차
prev_order_grp,,,
1–2번,894,-0.009,0.016
3–5번,1274,-0.019,0.013
6–9번,1415,-0.006,0.012


주자 배치별 직구 실제 − 예측


rs,000,001,010,011,100,101,110,111
n,7226.000,282.000,910.000,246.000,2886.000,503.000,1036.000,369.000
평균,0.008,-0.080,-0.003,-0.016,-0.006,-0.045,-0.016,-0.002
표준오차,0.005,0.029,0.016,0.030,0.009,0.021,0.015,0.025


#### 0-1-2. 위치: 주자 상황 vs 주자 신원

- 위치: 실전 24칸 확률로 묶은 H/S/C 예측 vs 실제
- 그룹: 주자 상황 (없음 / 1루만 / 득점권), 1루 주자(직전 타자)의 타순

In [4]:
d["상황"] = np.select([~runner, d["base1"].eq(1) & d["base2"].eq(0) & d["base3"].eq(0), (d["base2"] + d["base3"]) > 0],
                      ["주자 없음", "1루만", "득점권"], "기타")
d["1루 주자 타순"] = np.where(d["prev_base"].eq(1), d["prev_order_grp"], np.where(runner, "모름·다른 루", "주자 없음"))
G3 = d[fc.P_ZONE].to_numpy() @ fc.GROUP2  # 실전 H/S/C 확률 (n, 3)
for i, gname in enumerate(fc.HSC):
    d["r_" + gname] = (d["zone_group"] == gname).astype(float) - G3[:, i]  # 실제 − 예측

def loc_tab(by):  # 그룹별 H/S/C 실제 − 예측
    g = d.groupby(by)
    t = g[["r_H", "r_S", "r_C"]].mean().set_axis(["H 실제−예측", "S 실제−예측", "C 실제−예측"], axis=1)
    t.insert(0, "n", g.size())
    t["C 표준오차"] = g["r_C"].std() / np.sqrt(g.size())
    return t

print("위치 H/S/C 실제 − 예측 (실전)")
display(loc_tab("상황").round(3), loc_tab("1루 주자 타순").round(3))

위치 H/S/C 실제 − 예측 (실전)


,n,H 실제−예측,S 실제−예측,C 실제−예측,C 표준오차
상황,,,,,
1루만,2886,0.019,0.007,-0.026,0.009
득점권,3346,-0.029,-0.007,0.036,0.009
주자 없음,7226,0.024,0.020,-0.043,0.006


,n,H 실제−예측,S 실제−예측,C 실제−예측,C 표준오차
1루 주자 타순,,,,,
1–2번,894,0.017,-0.007,-0.010,0.016
3–5번,1274,-0.023,0.012,0.011,0.014
6–9번,1415,0.012,-0.005,-0.008,0.013
모름·다른 루,2661,-0.017,-0.002,0.019,0.010
주자 없음,7214,0.024,0.020,-0.043,0.006


### 🔍 0-1 결과
#### 주자 신원은 구종·위치 모두 신호 없음 
#### -> 적용 안 함 / 주자 상황은 위치에 큰 신호 (3단계)
- 구종: model_05 잔차에 주자 칸을 더해도 이득 없음 (주자 배치 × 아웃 +0.6, 직전 타자 루 × 타순 −0.1, 모두 95% 구간에 0 포함) -> model_05가 주자 몫(0-2에서 5.3)을 이미 가져감
- 간이 복원 주자 있는 초구의 66.5% (대부분 1루), 1루 주자 타순별 직구 잔차 차이 없음 (−0.006 ~ −0.019, 표준오차 0.012–0.016)
- 위치: 존 밖(C) 실제 − 예측 득점권 +0.036, 주자 없음 −0.043 -> 위치에서 찾은 가장 큰 신호 (3단계), 1루 주자 타순별로는 차이 없음
- 타순은 주루 능력의 약한 대용 -> 주루 능력을 직접 재는 변수가 생기면 다시

## 모델

## 1. 파라미터

- 구종 B4·위치: model_03 최종값
- LightGBM: 2장에서 고름

In [5]:
# ① 구종 출발점 B4: model_05 값 (대조군) -> 2-1에서 시즌 초 배수·기준점과 함께 다시 고름
M09_B4 = dict(half_life=25.0, alpha=5.48, hand_k=5.48, ref="all", season_f=1.0)
# ② 위치 (model_03, 고정)
LOC_REF, LOC_REF_HL = "decay", 173.0
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
ZONE_M = 32
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4(시즌 초 배수) + LightGBM (A 이번 타석 상황)

### 2-1. B4 다시 고르기: 시즌 초 배수·기준점 (2019, B4만)

- `type_online(…, ref, season_f)`: model_05 B4에 두 가지를 더함 (`ref="all"`, `season_f=1`이면 model_05와 같음 -> 재현 확인)
  - `season_f`: `fc.prior_table`·`fc.lookup`의 시즌 초 배수 (fp_common, model_06부터) -> 새 시즌 첫 경기 전에 지난 시즌까지 누적 S·N × season_f
  - `ref`: 기준점 `all` 지난 시즌 전부 / `last` 직전 시즌만 / `equal` 지난 시즌 투수 동일 가중 / `decayH` 전날까지 반감기 H(리그 초구 수)
- 거친 격자 (반감기 3 × season_f 4 × 기준점 6 × alpha 5 × hand_k 3) -> 1등 반감기가 범위 끝(35.4) -> 넓힌 세밀한 격자 (반감기 5 × season_f 7 × 기준점 2 × alpha 5 × hand_k 5)
- 기준점 선택: 1등 기준점과 지금(`all`)의 1등 차이가 경기 단위 부트스트랩 95% 구간에 0을 포함하면 지금 유지 (평평한 구간 규칙)

In [6]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

In [7]:
FP_ALL = fp.assign(_all=1)  # 리그 전체 한 그룹 (반감기 기준점용, 한 번만 만들어 prior_table 캐시가 같은 객체를 쓰게)

def league_ref(q, year, ref="all"):  # 기준점 (n, 9): "all" 지난 시즌 전부 (model_05) / "last" 직전 시즌만 / "equal" 투수 동일 가중 / "decayH" 전날까지 반감기 H
    if year == fp["GYEAR"].min():  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
        return np.broadcast_to(fp.loc[fp["GYEAR"] == year, CLASSES].mean().to_numpy(), (len(q), len(CLASSES)))
    past = fp[fp["GYEAR"] < year]
    if ref.startswith("decay"):
        S, N = fc.lookup(fc.prior_table(FP_ALL, ["_all"], CLASSES, float(ref[5:])), q.assign(_all=1), ["_all"], CLASSES)
        return S / np.maximum(N[:, :1], 1e-12)  # 원-핫이라 9개 열 N이 같음 -> 첫 열만
    v = {"all": lambda: past[CLASSES].mean(), "last": lambda: fp.loc[fp["GYEAR"] == year - 1, CLASSES].mean(),
         "equal": lambda: past.groupby("pitcher_code")[CLASSES].mean().mean()}[ref]()
    return np.broadcast_to(v.to_numpy(), (len(q), len(CLASSES)))

def type_sn(q, half_life, season_f):  # 투수별·투수×손별 전날까지 S, N (시즌 초 배수 반영)
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    return out

def type_online(q, year, half_life, alpha, hand_k, ref="all", season_f=1.0):  # B4 구종 확률 (n, 9): 기준점 -> 투수 (alpha) -> 투수×손 (hand_k)
    S, N, S2, N2 = type_sn(q, half_life, season_f)
    return fc.shrink(S2, N2, fc.shrink(S, N, league_ref(q, year, ref), alpha), hand_k)

m03 = pd.read_parquet("results/model_03/per_pitch.parquet")  # model_03·05 B4 확률 (재현 확인용)
m05 = pd.read_parquet("results/model_05/per_pitch.parquet")  # 비교 대상
for yr, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    assert np.allclose(type_online(q, yr, **M09_B4), fc.align(m03, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-9), "B4 재현 실패"
print("ref='all', season_f=1 -> model_05 B4와 같음 (재현 확인, fp_common prior_table 기본값 그대로)")

yv = y_type[val]
nll_v = lambda P: -np.log(np.clip(P[np.arange(len(yv)), yv], 1e-12, None))  # 2019 공별 log loss
_SN, _REF = {}, {}

def score_b4(half_life, season_f, ref, alpha, hand_k):  # 2019 B4 log loss (lookup·기준점은 캐시)
    if (half_life, season_f) not in _SN:
        _SN[(half_life, season_f)] = type_sn(q_val, half_life, season_f)
    if ref not in _REF:
        _REF[ref] = league_ref(q_val, fc.VAL_YEAR, ref)
    S, N, S2, N2 = _SN[(half_life, season_f)]
    return {"logloss": nll_v(fc.shrink(S2, N2, fc.shrink(S, N, _REF[ref], alpha), hand_k)).mean()}

GRID_B4 = {"half_life": [17.7, 25.0, 35.4], "season_f": [1.0, 0.5, 0.25, 0.1],
           "ref": ["all", "last", "equal", "decay1000", "decay3000", "decay10000"], "alpha": [1.0, 3.0, 5.48, 10.0, 30.0], "hand_k": [3.0, 5.48, 10.0]}
GRID_B4F = {"half_life": [25.0, 35.4, 50.0, 70.7, 100.0], "season_f": [1.0, 0.15, 0.2, 0.25, 0.3, 0.35, 0.5],
            "ref": ["all", "last"], "alpha": [3.0, 4.0, 5.48, 7.5, 10.0], "hand_k": [3.0, 4.0, 5.48, 7.5, 10.0]}  # 반감기 범위 끝(35.4) 넓힘 + 1등 주변
res_b4 = pd.concat([grid_search(score_b4, GRID_B4), grid_search(score_b4, GRID_B4F)], ignore_index=True)
res_b4 = res_b4.drop_duplicates(list(GRID_B4)).sort_values("logloss").reset_index(drop=True)
b4_now = score_b4(**{k: M09_B4[k] for k in GRID_B4})["logloss"]
print(f"2019 B4 log loss: model_05 {b4_now:.5f} -> 1등 {res_b4.logloss.iloc[0]:.5f} ({res_b4.logloss.iloc[0] - b4_now:+.5f})")
display(res_b4.head(8).round(5))
GRID_U = {k: sorted(set(GRID_B4[k]) | set(GRID_B4F[k]), key=lambda v: (isinstance(v, str), v)) for k in GRID_B4}  # 두 격자 후보 합침
for k, v in marginal(res_b4, GRID_U).items():  # 파라미터별 값마다 1등 log loss (민감도)
    print(k, v.round(5).to_dict())

ref='all', season_f=1 -> model_05 B4와 같음 (재현 확인, fp_common prior_table 기본값 그대로)
2019 B4 log loss: model_05 1.09815 -> 1등 1.09338 (-0.00477)


,half_life,season_f,ref,alpha,hand_k,logloss
0,35.4,0.25,last,7.50,5.48,1.09338
1,35.4,0.30,last,7.50,5.48,1.09345
2,35.4,0.25,last,5.48,7.50,1.09348
3,35.4,0.25,last,5.48,5.48,1.09348
4,35.4,0.30,last,5.48,7.50,1.09351
5,35.4,0.25,last,10.00,5.48,1.09351
6,35.4,0.25,equal,10.00,5.48,1.09351
7,35.4,0.30,last,10.00,5.48,1.09353


half_life {17.7: 1.09961, 25.0: 1.09495, 35.4: 1.09338, 50.0: 1.0936, 70.7: 1.09474, 100.0: 1.09623}
season_f {0.1: 1.09506, 0.15: 1.09383, 0.2: 1.09353, 0.25: 1.09338, 0.3: 1.09345, 0.35: 1.09363, 0.5: 1.0945, 1.0: 1.09779}
ref {'all': 1.09367, 'decay1000': 1.09393, 'decay10000': 1.09369, 'decay3000': 1.09375, 'equal': 1.09351, 'last': 1.09338}
alpha {1.0: 1.0963, 3.0: 1.09392, 4.0: 1.09361, 5.48: 1.09348, 7.5: 1.09338, 10.0: 1.09351, 30.0: 1.09475}
hand_k {3.0: 1.09433, 4.0: 1.0937, 5.48: 1.09338, 7.5: 1.09348, 10.0: 1.09394}


In [8]:
# 기준점 고르기: 1등 기준점 vs 지금(all) 각각의 1등, 공별 차이를 경기 단위 부트스트랩
def b4_row(cond):  # 조건을 만족하는 행 중 1등 (Series)
    return res_b4[cond].iloc[0]

def b4_probs(r, q=q_val, year=fc.VAL_YEAR):  # 표 한 행의 파라미터로 B4 확률
    return type_online(q, year, r.half_life, r.alpha, r.hand_k, r.ref, r.season_f)

GID_V = q_val["gameID"].to_numpy()
def game_ci(dif, B=1000):  # 경기 단위 부트스트랩 95% 구간 (공별 차이)
    s = pd.DataFrame({"g": GID_V, "d": dif}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

top, top_all = res_b4.iloc[0], b4_row(res_b4["ref"].eq("all"))
dif = nll_v(b4_probs(top)) - nll_v(b4_probs(top_all))
lo, hi = game_ci(dif)
print(f"1등 기준점 {top.ref} vs all: {dif.mean():+.5f} (95% {lo:+.5f} ~ {hi:+.5f})")
B4_P = top if hi < 0 else top_all  # 1등 기준점이 확실히 좋을 때만 바꿈
B4_P = dict(half_life=float(B4_P.half_life), alpha=float(B4_P.alpha), hand_k=float(B4_P.hand_k), ref=str(B4_P.ref), season_f=float(B4_P.season_f))
print("고른 B4:", B4_P)

# 분해: 반감기만 / 시즌 초만 / 둘 다 (기준점 all, 나머지는 그 조건 1등)
is_all = res_b4["ref"].eq("all")
rows = {}
for nm, cond in {"model_05 (반감 25, season_f 1)": is_all & res_b4.half_life.eq(25) & res_b4.season_f.eq(1) & res_b4.alpha.eq(5.48) & res_b4.hand_k.eq(5.48),
                 "반감기·alpha·hand_k만 (season_f 1)": is_all & res_b4.season_f.eq(1),
                 "시즌 초 배수만 (반감 25)": is_all & res_b4.half_life.eq(25),
                 "둘 다": is_all}.items():
    r = b4_row(cond)
    rows[nm] = {"반감기": r.half_life, "season_f": r.season_f, "alpha": r.alpha, "hand_k": r.hand_k, "2019 log loss": r.logloss, "model_05 대비": r.logloss - b4_now}
display(pd.DataFrame(rows).T.round(5))

# 시기별 이득: 투수의 그 시즌 몇 번째 경기인지, 월
P_new_v = type_online(q_val, fc.VAL_YEAR, **B4_P)
gain = nll_v(type_online(q_val, fc.VAL_YEAR, **M09_B4)) - nll_v(P_new_v)  # 양수 = 좋아짐
qq = q_val.assign(g=gain, 경기순=q_val.groupby("pitcher_code")["game_date"].rank(method="dense"))
qq["시기"] = pd.cut(qq["경기순"], [0, 1, 3, 5, 10, 999], labels=["시즌 첫 경기", "2–3번째", "4–5번째", "6–10번째", "11번째+"])
t = qq.groupby("시기", observed=True)["g"].agg(초구="size", 이득_x1000=lambda x: x.mean() * 1000)
t["전체 이득 중 몫"] = qq.groupby("시기", observed=True)["g"].sum() / gain.sum()
display(t.round(3))
display(qq.groupby(qq["game_date"].dt.month)["g"].agg(초구="size", 이득_x1000=lambda x: x.mean() * 1000).round(2).T)

1등 기준점 last vs all: -0.00029 (95% -0.00091 ~ +0.00014)
고른 B4: {'half_life': 35.4, 'alpha': 7.5, 'hand_k': 5.48, 'ref': 'all', 'season_f': 0.25}


,반감기,season_f,alpha,hand_k,2019 log loss,model_05 대비
"model_05 (반감 25, season_f 1)",25.0,1.00,5.48,5.48,1.09815,0.00000
반감기·alpha·hand_k만 (season_f 1),25.0,1.00,7.50,5.48,1.09810,-0.00005
시즌 초 배수만 (반감 25),25.0,0.35,5.48,5.48,1.09525,-0.00290
둘 다,35.4,0.25,7.50,5.48,1.09367,-0.00448


,초구,이득_x1000,전체 이득 중 몫
시기,,,
시즌 첫 경기,432,24.016,0.172
2–3번째,950,1.043,0.016
4–5번째,956,3.149,0.050
6–10번째,2493,7.618,0.315
11번째+,8627,3.126,0.447


game_date,3,4,5,6,7,8,9,10
초구,703.00,1865.00,2403.00,2493.00,1791.00,2362.00,1794.00,47.00
이득_x1000,18.19,11.59,4.12,-1.35,4.18,4.57,0.73,-4.13


### 🥇1등🥇 B4 (시즌 초 배수)
#### 2019 B4 log loss 1.0982 -> 1.0937 (−0.0045): 시즌 초 배수 0.25, 반감기 35.4, alpha 7.5, hand_k 5.48, 기준점 지난 시즌 전부 (그대로)
- **시즌 초 배수가 핵심**: 새 시즌 첫 경기 전에 지난 시즌까지 누적 무게를 1/4로 -> 0.2–0.35는 거의 같음, 1(지금) 1.0978 · 0.5 1.0945 · 0.1 1.0951
- 분해: 반감기·alpha만 바꾸면 −0.0001, 시즌 초 배수만 −0.0029, 함께 −0.0045 -> 지난 시즌 꼬리를 끊으니 시즌 안 기록을 더 길게 보는 게(반감 25 -> 35.4) 이득이 됨
- 이득 위치: 시즌 첫 경기 +0.024 (전체 이득의 17%), 3월 +0.018 · 4월 +0.012, 6–10번째 경기·11번째 이후도 조금씩 (긴 반감기 몫)
- **기준점은 효과 없음**: 1등(직전 시즌만) vs 지금 −0.0003 (95% −0.0009 ~ +0.0001) -> 지금 유지, 전날까지 반감기(1000–10000)·투수 동일 가중도 지금과 같거나 나쁨
- 범위: 반감기 1등 35.4가 처음 격자 끝 -> 50·70.7·100까지 넓혀 확인 (50 1.0936, 100 1.0962), 나머지도 범위 안쪽

### 2-2. A 이번 타석 상황 변수

- `fc.situation_features` (fp_common): 모든 공(구종 의심 공 포함) 기준, 경기 기록만, 그 공 앞 정보
- `pc` 그 경기 그 투수의 앞선 투구 수 (0 = 그 경기 첫 타자) / `tto` 같은 타자 몇 번째 / `starter` 그 경기 첫 공이 1회 / `sdiff` 투수 팀 기준 점수 차
- 미래 정보 검사 `fc.check_no_future`: 공 300개(그 경기 첫 타자 60개 포함)마다 그 공 직전까지 자른 데이터(`fc.as_of`)로 다시 계산 -> 전체 데이터 값과 같아야 함
  - 자른 데이터: 전날까지 전부 / 오늘 다른 경기·같은 경기 앞 공은 경기 기록만 (트래킹 비움) / 그 공은 투구 전 상태만 (결과·트래킹 비움)
  - 검사가 잡는지 확인: 일부러 미래를 쓴 변수(그 경기 총 투구 수 99/100, 그 공 구종 47/100 = 직구일 때만 다름)는 잡힘 (스크래치 확인)

In [9]:
A_COLS = fc.A_COLS
X_all = fc.situation_features(m, fp)
assert X_all.notna().all().all()
display(X_all.describe().T[["mean", "min", "max"]].round(2))
bad = fc.check_no_future(fc.situation_features, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=300)
assert bad.empty, bad

,mean,min,max
pc,33.39,0.0,124.0
tto,1.59,1.0,5.0
starter,0.69,0.0,1.0
inning,4.55,1.0,12.0
out_cn,0.97,0.0,2.0
base1,0.36,0.0,1.0
base2,0.19,0.0,1.0
base3,0.10,0.0,1.0
batorder,4.81,1.0,9.0
sdiff,0.37,-21.0,19.0


미래 정보 검사: 300개 중 다른 공 0개


### 2-3. LightGBM 다시 고르기 (2019)

- B4 출발점이 바뀌었으므로 연결된 LightGBM 파라미터만 다시 고름 (A 변수·learning_rate 등은 그대로)
- model_05 1등 주변 작은 격자 (잎 7·15 × 잎당 최소 20·50·200 × lambda_l2 300·1000·3000), model_05 값(7, 20, 1000) 포함 = 대조군
- 1등이 범위 끝이면 바깥으로 넓혀 다시 (lambda_l2 ×3, 잎 ×2, 최대 2번)
- 선택 규칙 model_05와 같음: 2019 1등과 0.0002 안이면 덜 극단적인 조합 (잎 적게 -> 트리 적게)

In [10]:
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = B4_P["half_life"], B4_P["alpha"], B4_P["hand_k"]  # 이름은 model_05와 같게 (아래 코드 그대로 쓰려고)
K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020 B4 (새 값)
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = type_online(fp[k], yr, **B4_P)
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05와 같음

def fit_type(train_years, lp, rounds=None, val_year=None):  # 학습 해로 LightGBM 학습 (rounds 없으면 val_year로 조기 종료)
    tr = np.flatnonzero(fp["GYEAR"].between(*train_years).to_numpy())
    dtr = lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr], free_raw_data=False)
    if rounds is None:
        va = np.flatnonzero(fp["GYEAR"].eq(val_year).to_numpy())
        dva = lgb.Dataset(X_all.iloc[va], y_all[va], init_score=INIT[va], reference=dtr)
        return lgb.train({**BASE, **lp}, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train({**BASE, **lp}, dtr, rounds)

def type_prob(bst, idx, rounds=None):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    raw = bst.predict(X_all.iloc[idx], raw_score=True, num_iteration=rounds or bst.best_iteration) + INIT[idx]
    return np.exp(log_softmax(raw, axis=1))

iv = np.flatnonzero(val)
def score_lgb(num_leaves, min_data_in_leaf, lambda_l2):  # LightGBM 한 조합의 2019 log loss, 트리 수
    lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
    bst = fit_type((TRAIN_FIRST, fc.VAL_YEAR - 1), lp, val_year=fc.VAL_YEAR)
    return {"트리 수": bst.best_iteration, "logloss": fc.prob_metrics(type_prob(bst, iv), y_type[val])["logloss"]}

GRID_LGB = {"num_leaves": [7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
res_lgb = grid_search(score_lgb, GRID_LGB)
for rnd in range(2):  # 1등이 범위 끝이면 넓힘
    b = res_lgb.iloc[0]
    add = {}
    if b.lambda_l2 == res_lgb.lambda_l2.max():
        add["lambda_l2"] = [b.lambda_l2 * 3]
    if b.num_leaves == res_lgb.num_leaves.max():
        add["num_leaves"] = [int(b.num_leaves * 2)]
    if not add:
        break
    cand = {"num_leaves": [int(b.num_leaves)], "min_data_in_leaf": [int(b.min_data_in_leaf)], "lambda_l2": [float(b.lambda_l2)], **add}
    print(f"{rnd + 1}번째 넓히기:", add)
    res_lgb = pd.concat([res_lgb, grid_search(score_lgb, cand)]).drop_duplicates(list(GRID_LGB)).sort_values("logloss").reset_index(drop=True)
TOL = 2e-4
ok = res_lgb[res_lgb["logloss"] <= res_lgb["logloss"].iloc[0] + TOL]
pick = ok.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 덜 극단적인 값: 잎 적게 -> 트리 적게
LGB_P = {"num_leaves": int(pick.num_leaves), "min_data_in_leaf": int(pick.min_data_in_leaf), "lambda_l2": float(pick.lambda_l2)}
LGB_ROUNDS = int(pick["트리 수"])
ctl = res_lgb[(res_lgb.num_leaves == 7) & (res_lgb.min_data_in_leaf == 20) & (res_lgb.lambda_l2 == 1000)].iloc[0]
b4_ll = fc.prob_metrics(P_B4[iv], y_type[val])["logloss"]
print(f"새 B4 2019 log loss {b4_ll:.4f} / LightGBM 1등 {res_lgb.logloss.iloc[0]:.4f} / 대조군(model_05 값) {ctl.logloss:.4f} / "
      f"선택 {pick.logloss:.4f} {LGB_P}, 트리 {LGB_ROUNDS}개")
display(res_lgb.head(10).round(5))

1번째 넓히기: {'num_leaves': [30]}
새 B4 2019 log loss 1.0937 / LightGBM 1등 1.0623 / 대조군(model_05 값) 1.0623 / 선택 1.0624 {'num_leaves': 7, 'min_data_in_leaf': 200, 'lambda_l2': 1000.0}, 트리 730개


,num_leaves,min_data_in_leaf,lambda_l2,트리 수,logloss
0,15,50,1000.0,613,1.06226
1,7,50,1000.0,795,1.06227
2,7,20,3000.0,1789,1.06230
3,15,200,1000.0,622,1.06230
4,7,20,1000.0,794,1.06230
5,7,50,3000.0,1791,1.06231
6,15,20,3000.0,1475,1.06232
7,15,20,1000.0,622,1.06232
8,15,50,3000.0,1471,1.06233
9,15,200,3000.0,1475,1.06235


### 🥇1등🥇 LightGBM
#### 새 B4 위에서도 model_05 값과 거의 같음
#### 잎 7, 잎당 최소 200, lambda_l2 1000, 트리 730개 (2019 1.0624)
- 1등(잎 15·잎당 50) 1.06226, model_05 값(잎 7·잎당 20·λ 1000) 1.06230 -> 차이 0.00004, 평평한 구간 규칙으로 잎 7·트리 적은 조합
- 잎 1등이 범위 끝(15) -> 30으로 넓혀 확인, 나아지지 않음
- A 위 LightGBM 이득은 그대로 (B4 1.0937 -> 1.0624, −0.031) -> 시즌 초 이득과 상황 이득이 서로 겹치지 않음

### 2-4. 최종 학습과 구종 확률

- 2019 예측: 2017–18 학습 모델 / 2020 예측: 같은 파라미터·트리 수로 2017–19 다시 학습

In [11]:
BST = {fc.VAL_YEAR: fit_type((TRAIN_FIRST, fc.VAL_YEAR - 1), LGB_P, LGB_ROUNDS),
       fc.TEST_YEAR: fit_type((TRAIN_FIRST, fc.TEST_YEAR - 1), LGB_P, LGB_ROUNDS)}
P_TYPE_ALL = np.full((len(fp), K), np.nan)
for yr, bst in BST.items():
    idx = np.flatnonzero(fp["GYEAR"].eq(yr).to_numpy())
    P_TYPE_ALL[idx] = type_prob(bst, idx, LGB_ROUNDS)
P09 = {yr: fc.align(m05, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_05 구종 확률 (fp 그 해 순서)
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("새 B4", P_B4[msk]), ("model_05", P09[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
imp = pd.DataFrame({yr: pd.Series(b.feature_importance("gain"), index=A_COLS) for yr, b in BST.items()})
display((imp / imp.sum()).sort_values(fc.VAL_YEAR, ascending=False).round(3).rename_axis("변수 (gain 비율)"))

log loss    top1    top2
2019 새 B4        1.0937  0.5279  0.8159
     model_05    1.0672  0.5343  0.8156
     model_06    1.0624  0.5360  0.8163
2020 새 B4        1.0916  0.5264  0.8179
     model_05    1.0610  0.5306  0.8207
     model_06    1.0574  0.5330  0.8203

,2019,2020
변수 (gain 비율),,
pc,0.295,0.285
out_cn,0.131,0.136
sdiff,0.115,0.111
batorder,0.101,0.089
inning,0.097,0.095
base2,0.077,0.087
base1,0.071,0.068
tto,0.062,0.069
base3,0.042,0.049


### 2-5. 진단 (2019)

- 시기별: 투수의 그 시즌 몇 번째 경기인지·월별 구종 log loss (model_05 vs model_06) -> 시즌 초 배수가 시즌 초에서 효과를 내는지
- 상황별 직구 치우침·포함률 gap: model_05 2-5와 같은 표
- 구종 1등 보정 구간표 (README 4-8: 다음 모델마다 같은 표로 기록)

In [12]:
nll = lambda P, y: -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
dv = q_val.assign(m05=nll(P09[fc.VAL_YEAR], y_type[val]), m06=nll(P_TYPE_ALL[val], y_type[val]),
                  경기순=q_val.groupby("pitcher_code")["game_date"].rank(method="dense"))
dv["시기"] = pd.cut(dv["경기순"], [0, 1, 3, 5, 10, 999], labels=["시즌 첫 경기", "2–3번째", "4–5번째", "6–10번째", "11번째+"])
for by in ["시기", dv["game_date"].dt.month.rename("월")]:
    t = dv.groupby(by, observed=True)[["m05", "m06"]].mean()
    t.insert(0, "초구", dv.groupby(by, observed=True).size())
    t["차이"] = t["m06"] - t["m05"]
    display(t.rename(columns={"m05": "model_05", "m06": MODEL}).round(4))

,초구,model_05,model_06,차이
시기,,,,
시즌 첫 경기,432,1.1677,1.1434,-0.0243
2–3번째,950,1.0964,1.0950,-0.0014
4–5번째,956,1.0712,1.0679,-0.0033
6–10번째,2493,1.1240,1.1155,-0.0085
11번째+,8627,1.0421,1.0387,-0.0034


,초구,model_05,model_06,차이
월,,,,
3,703,1.1223,1.1041,-0.0183
4,1865,1.0693,1.0575,-0.0118
5,2403,1.0988,1.0936,-0.0052
6,2493,1.1309,1.1315,0.0006
7,1791,1.0663,1.0614,-0.0049
8,2362,1.0064,1.0023,-0.0041
9,1794,1.0016,1.0010,-0.0006
10,47,0.7553,0.7633,0.0080


In [13]:
def ctx_check(P, msk):
    d = fp[msk].assign(**{c: X_all.loc[msk, c] for c in ["pc", "tto", "starter", "sdiff"]})  # X_all은 fp와 같은 순서
    y = y_type[msk]
    s = fc.show_types(P)
    i = np.arange(len(y))
    d["rFAST"] = (y == fc.TYPE_IDX["FAST"]) - P[:, fc.TYPE_IDX["FAST"]]
    d["gap"] = s[i, y] - (P * s).sum(1)
    cols = {"앞선 투구 수": pd.cut(d["pc"], [0, 1, 16, 31, 61, 999], right=False).astype(str),
            "같은 타자 몇 번째": d["tto"].clip(upper=3).astype(str), "아웃": d["out_cn"].clip(upper=2).astype(str),
            "득점권": np.where(d["base2"] + d["base3"] > 0, "득점권", "아님"), "선발/불펜": np.where(d["starter"] == 1, "선발", "불펜")}
    out = []
    for nm, c in cols.items():
        t = d.groupby(np.asarray(c))[["rFAST", "gap"]].mean()
        t.index = pd.MultiIndex.from_product([[nm], t.index], names=["변수", "값"])
        out.append(t)
    return pd.concat(out)

display(pd.concat({"model_05": ctx_check(P09[fc.VAL_YEAR], val), MODEL: ctx_check(P_TYPE_ALL[val], val)}, axis=1).round(3))
for nm, P in [("model_05", P09[fc.VAL_YEAR]), (MODEL, P_TYPE_ALL[val])]:
    print(nm, "2019 구종 1등 ECE", round(fc.calib_report(P, y_type[val], CLASSES)[0].loc["1등 예측", "ECE"], 4))

model_05        model_06       
                        rFAST    gap    rFAST    gap
변수         값                                        
앞선 투구 수    [0, 1)       0.021  0.030    0.024  0.037
           [1, 16)     -0.000  0.014    0.003  0.014
           [16, 31)    -0.011 -0.013   -0.009 -0.007
           [31, 61)    -0.011 -0.024   -0.009 -0.021
           [61, 999)    0.001 -0.020    0.003 -0.018
같은 타자 몇 번째 1            0.003  0.009    0.006  0.012
           2           -0.011 -0.026   -0.010 -0.023
           3           -0.006 -0.022   -0.005 -0.019
아웃         0            0.013  0.006    0.015  0.007
           1           -0.010 -0.010   -0.008 -0.008
           2           -0.011 -0.012   -0.009 -0.007
득점권        득점권         -0.021 -0.017   -0.018 -0.013
           아님           0.004 -0.002    0.006  0.001
선발/불펜      불펜          -0.004  0.009   -0.001  0.013
           선발          -0.001 -0.012   -0.000 -0.009

model_05 2019 구종 1등 ECE 0.0161
model_06 2019 구종 1등 ECE 0.0123


In [14]:
# 구종 1등 보정 구간표 (2019·2020): model_05 vs model_06
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("model_05", P09[yr]), (MODEL, P_TYPE_ALL[msk])]:
        rep, tabs = fc.calib_report(P, y_type[msk], CLASSES)
        t = tabs["1등 예측"]
        print(f"{yr} {nm}: 1등 ECE {rep.loc['1등 예측', 'ECE']:.4f}")
        display(t[t["공수"] >= 30].round(3).T)

2019 model_05: 1등 ECE 0.0161


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,103.000,1833.000,3808.000,3448.000,2285.000,1308.000,551.000,122.000
예측평균,0.287,0.362,0.453,0.546,0.646,0.744,0.847,0.921
실제비율,0.252,0.369,0.439,0.528,0.626,0.737,0.882,0.967
차이,-0.035,0.008,-0.014,-0.019,-0.020,-0.007,0.035,0.047


2019 model_06: 1등 ECE 0.0123


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,91.000,1820.000,3876.000,3490.000,2269.000,1249.000,542.000,121.000
예측평균,0.285,0.361,0.452,0.546,0.646,0.744,0.846,0.924
실제비율,0.341,0.364,0.447,0.526,0.632,0.738,0.886,0.967
차이,0.056,0.003,-0.005,-0.020,-0.014,-0.006,0.040,0.043


2020 model_05: 1등 ECE 0.0206


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,225.000,1760.000,3434.000,3132.000,1998.000,1250.000,695.000,121.000
예측평균,0.273,0.362,0.451,0.547,0.646,0.746,0.841,0.923
실제비율,0.271,0.362,0.432,0.512,0.624,0.747,0.881,0.967
차이,-0.002,0.001,-0.020,-0.036,-0.022,0.002,0.039,0.044


2020 model_06: 1등 ECE 0.0168


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,230.000,1792.000,3504.000,3075.000,2003.000,1207.000,683.000,121.000
예측평균,0.270,0.364,0.452,0.547,0.645,0.746,0.842,0.923
실제비율,0.296,0.358,0.442,0.518,0.628,0.742,0.881,0.975
차이,0.026,-0.006,-0.010,-0.029,-0.017,-0.003,0.040,0.052


### 🔍 진단
#### 시즌 초 이득이 LightGBM 위에서도 남음, 보정도 좋아짐
- 시기별 구종 log loss (2019, model_06 − model_05): 시즌 첫 경기 −0.024, 6–10번째 −0.009, 11번째 이후 −0.003 / 3월 −0.018 · 4월 −0.012 · 5월 −0.005, 6월 +0.001 · 10월 +0.008(초구 47개)
- 구종 1등 ECE: 2019 0.016 -> 0.012, 2020 0.021 -> 0.017 -> model_05에서 늘었던 몫이 대부분 돌아옴
  - 구간 모양은 그대로: 중간(50–70%) 조금 과신, 높은 구간(80%+) 과소 확신 -> README 4-8대로 계속 기록만
- 상황별 직구 치우침·포함률 gap은 model_05와 거의 같음 (시즌 초 배수는 상황 정보와 무관)

## 3. ② 위치 (model_03과 같음)

In [16]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [17]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_05와 같음, 구종 확률만 2장 결과(`P_TYPE_ALL`)

In [18]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [19]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종과 무관한 열(실제 구종 기준 위치)은 model_05과 같아야 함
chk = pp.merge(m05, on=fc.KEYS, suffixes=("", "_03"))
same = ["ll_zone_true", "nll_coord_true", "dist_true"]
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_03"], equal_nan=True)]
assert len(chk) == len(pp) and not bad, f"model_05과 다른 열: {bad}"
print("실제 구종 기준 위치 = model_05 (재현 확인)")

(26073, 48)
실제 구종 기준 위치 = model_05 (재현 확인)


## 5. 평가

- 채점은 `fc.evaluate`, model_05과 나란히 / 경기 단위 부트스트랩 (model_05 − model_05, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [20]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m05_sum = eval_years(m05)
display(pd.concat({MODEL: summary, "model_05": m05_sum, "차이 (model_05 대비)": summary - m05_sum}, axis=1).round(4))

model_06          model_05          차이 (model_05 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0624   1.0574   1.0672   1.0610          -0.0048 -0.0036
구종 top1               0.5360   0.5330   0.5343   0.5306           0.0017  0.0025
구종 top2               0.8163   0.8203   0.8156   0.8207           0.0007 -0.0004
구종 top3               0.9375   0.9409   0.9357   0.9387           0.0019  0.0021
구종 macro F1           0.3192   0.3496   0.3238   0.3361          -0.0046  0.0134
24칸 logloss (실전)      2.9494   2.9457   2.9496   2.9455          -0.0003  0.0002
24칸 top1              0.1301   0.1339   0.1291   0.1338           0.0010  0.0001
24칸 top3              0.2897   0.2913   0.2902   0.2906          -0.0005  0.0007
H/S/C logloss         1.0748   1.0768   1.0748   1.0767          -0.0000  0.0001
H/S/C 정확도             0.3951   0.3883   0.3952   0.3889          -0.0001 -0.0006
24칸 logloss (실제 구종)   2.8933   2.8967   2.8933   2.8967           0.0000  0.0000
구종×칸 logloss          3.9556   3.9534   3.9605   3.9569          -0.0048 -0.0036
좌표 평균거리 cm           32.6861  31.8351  32.6913  31.8324          -0.0052  0.0026
좌표 NLL                9.3173   9.2663   9.3175   9.2660          -0.0002  0.0003

In [21]:
display(fc.bootstrap_diff(fc.pitch_scores(m05, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = model_06이 좋음 (b − a)
# 실전 출력: 누적 80% 보여 주기 (2020)
for nm, P_ in [("model_05", m05), (MODEL, pp)]:
    d = fc.align(P_, fp, fc.TEST_YEAR)
    Pt_, yt_ = d[fc.P_TYPE].to_numpy(), d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    s = fc.show_types(Pt_)
    print(f"{nm}: 보여 준 구종 수 평균 {s.sum(1).mean():.2f}, 포함률 {s[np.arange(len(yt_)), yt_].mean():.3f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,-0.0036,-0.0081,-0.0000,좋아짐
ll_zone,0.0002,-0.0002,0.0005,차이 불확실
ll_joint,-0.0036,-0.0081,-0.0000,좋아짐
nll_coord,0.0003,-0.0001,0.0007,차이 불확실
dist,0.0026,-0.0029,0.0077,차이 불확실


model_05: 보여 준 구종 수 평균 2.39, 포함률 0.902
model_06: 보여 준 구종 수 평균 2.40, 포함률 0.904


### 🥇1등🥇
#### 2020 구종 log loss 1.0610 -> 1.0574 (−0.0036), 부트스트랩 구종 좋아짐 (95% −0.0081 ~ −0.0000), 나빠진 점수 없음 -> 채택
- 2019 −0.0048 -> 2020 −0.0036 (B4만으로는 2019 −0.0045)
- 다른 점수는 차이 불확실: 24칸 실전 +0.0002, 좌표 거리 +0.003cm
- top1 0.531 -> 0.533, macro F1 0.336 -> 0.350, 누적 80% 보여 주기 포함률 90.2% -> 90.4% (평균 2.40개)
- **채택 -> 다음 비교 출발점** (사용자 확인)

## 6. 저장

- `results/model_06/` -> `compare.ipynb` 실행하면 자동 포함

In [ ]:
fc.save_results(MODEL, pp, {"summary": summary, "lgb_grid": res_lgb})

저장: results\model_06 ['lgb_grid.csv', 'per_pitch.parquet', 'summary.csv']
